In [ ]:
from paths import simplified_filepath, raw_export_filepath
from downloaders.LabelStudioInterface import LabelStudioInterface
from preprocessing.classes.helper_to_classes import get_image_path_from_task
from PIL import Image
from tqdm.auto import tqdm
from preprocessing.classes.AnnotatedPage import AnnotatedPage
from preprocessing.simplify.simplify_export import (
    simplify_export,
    load_simplified_export,
)
LSinterface = LabelStudioInterface(raw_export_filepath=raw_export_filepath)
LSinterface.save_export()
simplify_export(raw_export_filepath, simplified_filepath)
LS = load_simplified_export(simplified_filepath)


for task in tqdm(LS):
    task_id = task["id"]
    try:
        img_path = get_image_path_from_task(task)
        if img_path is None:
            print(
                f"(Task {task_id}) > No se ha podido encontrar el path a la imagen de esta tarea."
            )
            continue
        img = Image.open(img_path)
        for ann in task["annotations"]:
            try:
                Ann = AnnotatedPage(ann, task_id, img, unrotate=False, cc_ordering=True)
                if not (len(Ann.image_boxes) > 0):
                    print(f"La tarea {task_id} no tiene anotaciones.")
            except Exception as E:
                print(
                    f"Task {task_id} - Error durante la creación de la AnnotatedPage: {E}"
                )
                raise E
    except Exception as E:
        print(
            f"Task {task_id} - Error durante la creación de los parámetros de AnnotatedPag: {E}"
        )
        raise E

print(f"Procesadas todas las {len(LS)} tareas.")
